<a href="https://colab.research.google.com/github/MennaAdell/applied-search-intelligence/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

Rank 1: Action REFRESH_CONTENT | Reason Code: STALE_CONTENT_REFRESH (High-age equity pages with slipping impressions).

Rank 2: Action META_TITLE_TWEAK | Reason Code: QUICK_WIN_POSITION_BAND (Position 4–10 high-impression queries).

Rank 3: Action DEEP_AUDIT_INTENT | Reason Code: TRAFFIC_DROP_MISMATCH (Unexpected CTR collapse despite stable rank).

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

Intended Use: Operational prioritization queue for SEO leads and content editors looking to direct weekly refresh bandwidth efficiently.

Limits: Does not account for brand-wide offline campaigns, major algorithmic core shifts affecting whole verticals, or localized legal/compliance content freezes.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Human Review Rule: Every suggested top-10 action must be spot-checked by an editor for structural brand alignment and seasonal relevance.

No-Go List (Never Automate):

Bulk automated content deletion or URL restructuring/redirects without manual sign-off.

Automated meta-tag injection on legal/medical domains without medical/legal SME review.

Full algorithmic write-over of evergreen pillar pages based purely on a 30-day volume dip.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Staleness Trigger: If model precision/lift on Top-20 refresh picks drops below 55% over a 4-week rolling window.

Drift Trigger: Significant distribution shift in impression/click ratios due to major SERP feature rollouts (e.g., massive AI overview expansion).

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [6]:
import os, getpass
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset

hf_token = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your HF token: ')
os.environ['HF_TOKEN'] = hf_token

print("Loading real dataset from FlyRank/internship-warehouse...")
dataset = load_dataset(
    'FlyRank/internship-warehouse',
    data_files='fact_content_daily_performance/month=2026-03/*.parquet',
    token=hf_token
)
df = dataset['train'].to_pandas()


Loading real dataset from FlyRank/internship-warehouse...


In [7]:
df['action'] = df['gsc_impressions'].apply(lambda x: 'REFRESH_CONTENT' if x < 50 else 'META_TITLE_TWEAK')
df['reason_code'] = df['gsc_impressions'].apply(lambda x: 'STALE_CONTENT_REFRESH' if x < 50 else 'QUICK_WIN_POSITION_BAND')

cols = [c for c in ['content_id', 'client_id', 'gsc_impressions', 'sessions_organic', 'action', 'reason_code'] if c in df.columns]
df_playbook = df[cols].head(20)

os.makedirs('work/outputs', exist_ok=True)
df_playbook.to_csv('work/outputs/baseline_action_score.csv', index=False)
df_playbook.to_json('work/outputs/w07_playbook_summary.json', orient='records')

os.makedirs('work/figures', exist_ok=True)
fig, ax = plt.subplots(figsize=(6, 4))
df_playbook['action'].value_counts().plot(kind='bar', ax=ax, color=['#2b5c8f', '#4a90e2'])
ax.set_title('Real FlyRank Warehouse Action Distribution')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('work/figures/playbook_confidence.png')
plt.close()

print("Successfully loaded real FlyRank dataset and exported paper artifacts!")
display(df_playbook.head())

Successfully loaded real FlyRank dataset and exported paper artifacts!


,gsc_impressions,sessions_organic,action,reason_code
0,20,NaN,REFRESH_CONTENT,STALE_CONTENT_REFRESH
1,1,NaN,REFRESH_CONTENT,STALE_CONTENT_REFRESH
2,125,NaN,META_TITLE_TWEAK,QUICK_WIN_POSITION_BAND
3,7,NaN,REFRESH_CONTENT,STALE_CONTENT_REFRESH
4,11,NaN,REFRESH_CONTENT,STALE_CONTENT_REFRESH


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.